# Assignment 2: Data Cleaning

Group 33: Michael Massaad (300293612) & Gabriel Zohrob (300309391)

## Part 1: Validity Checker

## Part 2: Missing Data and Imputation 

### Description

Title: Dataset 2 – Imputation (Insurance / Medical Cost Personal Dataset)

Dataset name: Medical Cost Personal Dataset (Insurance)

Original source: Kaggle dataset (Insurance / Medical Cost Personal Dataset)

Shape: 7 columns, 1338 rows.

Purpose: Contains demographic and lifestyle attributes used to study/predict medical insurance charges (real-world tabular dataset, not synthetic).

Feature description:

- age: age of the primary beneficiary - numerical
- sex: male/female - categorical
- bmi: body mass index - numerical
- children: Number of children covered by health insurance / Number of dependents - numerical
- smoker: yes/no - categorical
- region: the beneficiary's residential area in the US, northeast, southeast, southwest, northwest.  - categorical
- charges: medical insurance charges - numerical

In [187]:
##!pip install scikit-learn

In [188]:
# ===============================
# Import Required Libraries
# ===============================

# NumPy: used for numerical operations and random number generation
import numpy as np

# Pandas: used for loading and manipulating tabular datasets
import pandas as pd


# ===============================
# Import Tools for Imputation & Evaluation
# ===============================

# These metrics will be used to quantitatively evaluate
# how close the imputed values are to the original values
from sklearn.metrics import mean_absolute_error, mean_squared_error

# LinearRegression will be used for regression-based imputation (Experiment 2)
from sklearn.linear_model import LinearRegression

# KNNImputer will be used for similarity-based imputation (Experiment 3)
from sklearn.impute import KNNImputer


# ===============================
# Reproducibility Setup
# ===============================

# Setting a fixed random seed ensures that
# the same rows are masked every time the notebook is run.
# This makes results reproducible for the TA.
SEED = 4142

# Create a random number generator using the seed
rng = np.random.default_rng(SEED)


# ===============================
# Evaluation Function
# ===============================

def eval_imputation(y_true, y_pred):
    """
    This function evaluates the quality of imputation.

    Parameters:
    - y_true: the original ground-truth values (before masking)
    - y_pred: the imputed values

    Returns:
    - MAE (Mean Absolute Error)
    - RMSE (Root Mean Squared Error)

    We use these metrics to quantify how far the imputed
    values are from the real values.
    """
    
    # MAE: average absolute difference between true and predicted values
    mae = mean_absolute_error(y_true, y_pred)
    
    # RMSE: square root of average squared differences
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return {"MAE": mae, "RMSE": rmse}

In [189]:
url = "https://raw.githubusercontent.com/michaelmassaad02/CSI4142_Assignment2/refs/heads/main/insurance.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
display(df.head())

print("\nMissing values in original dataset:")
display(df.isna().sum())

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Missing values in original dataset:


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

### Experiment 1 — MCAR Missingness on bmi with Median Imputation

In this experiment, missing values will be artificially introduced in the bmi attribute under a Missing Completely At Random (MCAR) mechanism.

MCAR means that the probability of a value being missing is independent of both observed and unobserved data. In this case, 5% of the rows will be randomly selected and their bmi values will be set to missing.

To handle the missing data, we will use median imputation, which replaces missing values with the median of the observed values. The median is chosen because it is robust to outliers and provides a simple baseline imputation method.

The performance of this imputation method will be evaluated quantitatively using MAE and RMSE, comparing the imputed values to the original ground-truth values.

In [190]:
# Create a fresh copy for Experiment 1
df_exp1 = df.copy()

# Save the original ground truth values
bmi_true = df_exp1["bmi"].copy()

# Select 5% of rows completely at random (MCAR)
missing_fraction = 0.05
masked_indices = rng.choice(df_exp1.index, 
                            size=int(len(df_exp1) * missing_fraction), 
                            replace=False)

# Introduce missing values
df_exp1.loc[masked_indices, "bmi"] = np.nan

# Check how many missing values were introduced
print("Number of missing values in bmi after masking:")
print(df_exp1["bmi"].isna().sum())

Number of missing values in bmi after masking:
66


In [191]:
# ===============================
# Detect missing values + Median Imputation
# ===============================

# 1) Detect missing values (before imputation)
missing_before = df_exp1["bmi"].isna().sum()
print("Missing bmi BEFORE imputation:", missing_before)

# 2) Compute median from the OBSERVED (non-missing) bmi values
bmi_median = df_exp1["bmi"].median()
print("Median bmi used for imputation:", bmi_median)

# 3) Impute: replace missing bmi with the median
df_exp1["bmi_imputed"] = df_exp1["bmi"].fillna(bmi_median)

# 4) Verify missing values are resolved in the imputed column
missing_after = df_exp1["bmi_imputed"].isna().sum()
print("Missing bmi AFTER imputation (in bmi_imputed):", missing_after)

# Optional: show a few examples of what changed (masked rows only)
print("\nExamples (masked rows): original bmi vs after imputation")
display(df_exp1.loc[masked_indices[:5], ["bmi", "bmi_imputed"]])

Missing bmi BEFORE imputation: 66
Median bmi used for imputation: 30.38
Missing bmi AFTER imputation (in bmi_imputed): 0

Examples (masked rows): original bmi vs after imputation


,bmi,bmi_imputed
479,NaN,30.38
612,NaN,30.38
314,NaN,30.38
1224,NaN,30.38
498,NaN,30.38


In [192]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true = bmi_true.loc[masked_indices]

# Imputed values (after imputation)
y_pred = df_exp1.loc[masked_indices, "bmi_imputed"]

metrics_exp1 = eval_imputation(y_true, y_pred)
print("Experiment 1 Evaluation Metrics (Median Imputation on BMI):")
print(f"MAE:  {metrics_exp1['MAE']:.3f}")
print(f"RMSE: {metrics_exp1['RMSE']:.3f}")


Experiment 1 Evaluation Metrics (Median Imputation on BMI):
MAE:  5.322
RMSE: 6.734


In this experiment, 5% of the bmi values were randomly removed under a Missing Completely At Random (MCAR) mechanism. Because the missingness was introduced through uniform random sampling, it is independent of both observed and unobserved variables in the dataset.

Median imputation was used to replace the missing values. This method is simple and robust to outliers but does not leverage relationships between variables.

The evaluation results show:

MAE ≈ 5.322

RMSE ≈ 6.734

This indicates that, on average, the imputed BMI values differ from the true values by about 5 units. While median imputation provides a reasonable baseline under MCAR, it does not account for correlations between BMI and other features such as age or charges, which may limit its accuracy.

### Experiment 2 — MAR Missingness on age with Regression Imputation

In this experiment, missing values will be artificially introduced in the age variable under a Missing At Random (MAR) mechanism.

Missing At Random (MAR) means that the probability of a value being missing depends on an observed variable in the dataset. In this case, individuals who are smokers (smoker = "yes") will have a higher probability of having missing age values. Therefore, missingness depends on an observed attribute (smoker) rather than on the value of age itself.

To impute the missing age values, a linear regression model will be trained using complete observations. The model will learn the relationship between age and other predictor variables such as bmi, children, charges, sex, region, and smoker. The trained model will then predict the missing ages.

The imputation quality will be evaluated quantitatively using MAE and RMSE on the artificially masked rows only.

In [193]:
# ===============================
# Create fresh copy for Experiment 2
# ===============================
df_exp2 = df.copy()

# Save original ground truth
age_true = df_exp2["age"].copy()

# 5% missing overall, but dependent on smoker (MAR)
missing_fraction = 0.05

# Get indices where smoker == "yes"
smoker_yes_indices = df_exp2[df_exp2["smoker"] == "yes"].index

# Select 5% of smoker==yes rows
masked_indices_exp2 = rng.choice(
    smoker_yes_indices,
    size=int(len(smoker_yes_indices) * missing_fraction),
    replace=False
)

# Introduce missing values
df_exp2.loc[masked_indices_exp2, "age"] = np.nan

# Check missing values
print("Total missing values in age after masking:")
print(df_exp2["age"].isna().sum())

print("\nMissing values among smokers (yes):")
print(df_exp2.loc[smoker_yes_indices, "age"].isna().sum())

Total missing values in age after masking:
13

Missing values among smokers (yes):
13


In [194]:
# ===============================
# Regression Imputation for age
# ===============================

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_exp2, drop_first=True)

# Separate rows with and without missing age
train_data = df_encoded[df_encoded["age"].notna()]
test_data = df_encoded[df_encoded["age"].isna()]

# Define predictors (all columns except age)
X_train = train_data.drop(columns=["age"])
y_train = train_data["age"]

X_test = test_data.drop(columns=["age"])

# Train regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict missing ages
age_predicted = model.predict(X_test)

# Create a new column for imputed values
df_exp2["age_imputed"] = df_exp2["age"]

# Fill missing age values with predictions
df_exp2.loc[df_exp2["age"].isna(), "age_imputed"] = age_predicted

# Check that missing values are resolved
print("Missing values after regression imputation:")
print(df_exp2["age_imputed"].isna().sum())

Missing values after regression imputation:
0


In [195]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true_exp2 = age_true.loc[masked_indices_exp2]

# Predicted/imputed values
y_pred_exp2 = df_exp2.loc[masked_indices_exp2, "age_imputed"]

metrics_exp2 = eval_imputation(y_true_exp2, y_pred_exp2)

print("Experiment 2 Evaluation Metrics (Regression Imputation on age):")
print(f"MAE:  {metrics_exp2['MAE']:.3f}")
print(f"RMSE: {metrics_exp2['RMSE']:.3f}")

Experiment 2 Evaluation Metrics (Regression Imputation on age):
MAE:  11.957
RMSE: 14.228


In this experiment, missing values in age were introduced under a MAR mechanism, where missingness depended on the observed variable smoker.

Regression imputation was applied using other features as predictors. However, the evaluation results show relatively high MAE and RMSE values (approximately 12–14 years). This suggests that age does not have strong predictive relationships with the other variables in the dataset. As a result, regression imputation does not significantly improve accuracy compared to simpler methods in this case.

This experiment highlights that the effectiveness of model-based imputation depends on the strength of correlations between variables.

### Experiment 3 — MNAR Missingness on charges with KNN (Similarity-Based) Imputation

In this experiment, missing values will be artificially introduced in the charges variable under a Missing Not At Random (MNAR) mechanism.

MNAR means that the probability of a value being missing depends on the value of the variable itself (the missingness is related to the unobserved value). To simulate MNAR, we will make high charges values more likely to be missing by selecting rows in the top 20% of charges and masking a portion of them.

To impute missing charges values, we use a similarity-based method: KNNImputer. KNN imputation fills missing values by finding the most similar rows (nearest neighbors) based on other numeric features and averaging their values.

The imputation quality will be evaluated quantitatively using MAE and RMSE on the artificially masked rows only.

In [196]:
# ===============================
# Create fresh copy for Experiment 3
# ===============================
df_exp3 = df.copy()

# Save original ground truth
charges_true = df_exp3["charges"].copy()

# MNAR setup:
# Top 20% charges group
top_group_threshold = df_exp3["charges"].quantile(0.80)
top_group_idx = df_exp3[df_exp3["charges"] >= top_group_threshold].index

# Mask 25% of the top group => overall missingness ~ 0.20 * 0.25 = 0.05 (5%)
mask_within_top_fraction = 0.25
masked_indices_exp3 = rng.choice(
    top_group_idx,
    size=int(len(top_group_idx) * mask_within_top_fraction),
    replace=False
)

# Introduce missing values in charges
df_exp3.loc[masked_indices_exp3, "charges"] = np.nan

print("Total missing values in charges after masking:", df_exp3["charges"].isna().sum())
print("Missing values within top-charges group:", df_exp3.loc[top_group_idx, "charges"].isna().sum())

Total missing values in charges after masking: 67
Missing values within top-charges group: 67


In [197]:
# ===============================
# KNN Imputation for charges
# ===============================

# Select only numeric columns for KNN
numeric_cols = ["age", "bmi", "children", "charges"]
df_numeric = df_exp3[numeric_cols]

# Initialize KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)

# Apply imputation
df_imputed_array = knn_imputer.fit_transform(df_numeric)

# Convert back to DataFrame
df_imputed = pd.DataFrame(df_imputed_array, columns=numeric_cols)

# Create new column for imputed charges
df_exp3["charges_imputed"] = df_imputed["charges"]

# Check missing values after imputation
print("Missing values after KNN imputation:")
print(df_exp3["charges_imputed"].isna().sum())

Missing values after KNN imputation:
0


In [198]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true_exp3 = charges_true.loc[masked_indices_exp3]

# Imputed values
y_pred_exp3 = df_exp3.loc[masked_indices_exp3, "charges_imputed"]

metrics_exp3 = eval_imputation(y_true_exp3, y_pred_exp3)

print("Experiment 3 Evaluation Metrics (KNN Imputation on charges):")
print(f"MAE:  {metrics_exp3['MAE']:.3f}")
print(f"RMSE: {metrics_exp3['RMSE']:.3f}")

Experiment 3 Evaluation Metrics (KNN Imputation on charges):
MAE:  20019.950
RMSE: 22530.898


In this experiment, missing values were introduced in charges under an MNAR mechanism by masking only high-charge observations. Since missingness depends on the value of the variable itself, this represents the most challenging missing-data scenario.

KNN imputation was applied using similarity between observations based on numeric features. However, the evaluation results show very large MAE and RMSE values (approximately 20,000–22,000). This indicates that KNN struggled to accurately reconstruct extreme charge values.

This experiment demonstrates that imputation methods perform significantly worse under MNAR conditions, especially when extreme values are removed.

### Conclusion — Part 2: Missing Data and Imputation

In this section, three different missing-data mechanisms were simulated and evaluated using distinct imputation methods.

- Experiment 1 (MCAR + Median Imputation) showed moderate error. Since missingness was completely random, simple median imputation provided a reasonable baseline. However, it ignored relationships between variables.

- Experiment 2 (MAR + Regression Imputation) demonstrated that model-based imputation does not always guarantee improved accuracy. Because age had weak predictive relationships with other features, regression imputation produced relatively high error.

- Experiment 3 (MNAR + KNN Imputation) resulted in the largest error values. Since missingness depended on high charges values (extreme cases), similarity-based imputation struggled to accurately reconstruct them. This highlights the difficulty of handling MNAR data.

Overall, the experiments show that imputation performance strongly depends on:

- The missing-data mechanism (MCAR vs MAR vs MNAR), and

- The strength of relationships between variables.

MNAR scenarios were the most challenging, while simple methods performed reasonably under MCAR conditions.

## Ressources:

- Medical Cost Personal Dataset (Insurance Dataset).
Available on Kaggle:
https://www.kaggle.com/datasets/mirichoi0218/insurance


- OpenAI. (2024). ChatGPT (GPT-4). https://chat.openai.com/
ChatGPT was used as a learning support tool to clarify concepts related to missing data mechanisms (MCAR, MAR, MNAR), imputation methods, and evaluation metrics. It also assisted in refining explanations and improving the clarity and structure of written sections.